# 🏦 Topic 6: RAG Foundations

Welcome to the last topic of Day 2! Today we are building the piece that makes the Barclays Product Knowledge Assistant actually *know* things.

## Learning Objectives

By the end of this topic you will be able to:

1. Explain what an embedding is and why semantic search beats keyword search for product Q&A
2. Call the OpenAI Embeddings API to convert text chunks into dense vectors
3. Store and query embeddings in ChromaDB using cosine similarity
4. Wire together a complete RAG pipeline: retrieve relevant chunks, augment the prompt, get a grounded answer

## What we are building today

The **retrieval layer** of the Barclays Product Knowledge Assistant.

By the end of Topic 6:

- BEFORE: customer asks a question, model guesses (hallucination risk)
- AFTER:  customer asks a question, relevant product docs are retrieved and injected into the prompt, model answers with grounded facts

In Topic 7 we add web search on top of this vector store retrieval.

## Section 0: Environment Setup

Let's install ChromaDB and make sure we have the right versions. The `numpy<2` pin is important: ChromaDB 1.x uses numpy internally and breaks with numpy 2.x.

In [ ]:
# chromadb 1.5.8 is the current stable release (April 2026).
# openai >=1.30.0 for the embeddings API (client.embeddings.create).
# numpy<2 is mandatory: chromadb 1.x breaks on numpy 2.x - pin this always.
!pip install -q "chromadb==1.5.8" "openai>=1.30.0" "numpy<2"

In [ ]:
import os
import getpass

from openai import OpenAI
import chromadb

# SageMaker session - we keep this around for the optional S3 stretch lab at the end.
import sagemaker
from sagemaker import get_execution_role

sess = sagemaker.Session()
role = get_execution_role()

print(f"SageMaker region: {sess.boto_region_name}")

# Load the OpenAI API key. getpass hides what you type so the key never lands in the notebook output.
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key and press Enter: ")

# Build the OpenAI client - same pattern we used in Topics 1 through 5.
client = OpenAI()

# Same model constant we have used since Topic 1.
MODEL = "gpt-4o"

# New for Topic 6: the embeddings model. Cheap, fast, 1536-dimensional output vectors.
EMBED_MODEL = "text-embedding-3-small"

print(f"Chat model:      {MODEL}")
print(f"Embedding model: {EMBED_MODEL}")
print("Ready. client and chromadb imported.")

## What Are We Building Today?

Right now our Product Knowledge Assistant has a problem.

**The problem**: when a customer asks about a Barclays product, the model either makes something up (hallucination) or we manually paste in the right product text (not scalable). We need a way to automatically find and inject the right context.

**The solution**: Retrieval-Augmented Generation (RAG).

Here is the before/after we are working toward:

```python
# BEFORE (no RAG - what we have from Topics 3-5):
customer_question = "Does the Barclays Rewards card have a foreign transaction fee?"
answer = ask_chatbot(customer_question)   # model guesses - may be wrong

# AFTER (with RAG - end of this topic):
customer_question = "Does the Barclays Rewards card have a foreign transaction fee?"
relevant_chunks = retrieve(customer_question, collection, n_results=3)
answer = rag_answer(customer_question, relevant_chunks)   # grounded in real docs
```

We will build this in three steps:

1. Learn what embeddings are and how to create them
2. Build a ChromaDB vector store and add our document chunks
3. Wire together the full RAG pipeline

In [ ]:
# These are the document chunks we will embed and retrieve from.
# In Topic 2 you extracted and chunked real Barclays PDFs into a list called `chunks`.
# Here we define a small inline set so this notebook works even if you skipped Topic 2.
# In the Tier 3 lab at the end you can swap in your Topic 2 chunks for better coverage.

BARCLAYS_DOCS = [
    "Barclays Personal Loan: borrow 1,000 GBP to 35,000 GBP. Representative APR 6.5 percent for loans of 7,500 GBP to 15,000 GBP. Terms from 1 to 5 years. No arrangement fee. Early repayment allowed with 30-day interest.",
    "Barclays Rewards Credit Card: 0.25 percent cashback on all purchases. No annual fee for the first year, then 24 GBP per year. Foreign transaction fee: 0 percent. Minimum repayment is the greater of 25 GBP or 2.5 percent of the balance.",
    "Barclays Savings Account: instant access, variable rate currently 3.75 percent AER. No minimum deposit. Withdrawals allowed any time with no penalty. Interest credited monthly.",
    "Barclays Mortgage: fixed-rate deals from 2 to 10 years. Rates start at 4.2 percent for 60 percent LTV. Overpayments up to 10 percent of outstanding balance per year at no charge. Early repayment charge applies if you exceed this.",
    "Barclays Student Account: 0 percent interest overdraft up to 500 GBP in year 1, up to 1,000 GBP from year 2 onwards. No monthly fee. Includes a free Barclays debit card.",
    "Barclays Business Current Account: free banking for the first 12 months for new businesses. Monthly fee 8 GBP thereafter. Includes online banking and access to a dedicated relationship manager for accounts over 100,000 GBP.",
    "Barclays Travel Pack: includes travel insurance, airport lounge access, and zero foreign transaction fees. Monthly fee 18 GBP. Covers the account holder plus a named partner.",
]

# We also keep the variable name `chunks` for compatibility with Topic 2 patterns.
chunks = BARCLAYS_DOCS

print(f"Loaded {len(chunks)} document chunks.")
for i, c in enumerate(chunks):
    print(f"  [{i}] {c[:70]}...")

## Concept 1: Embeddings - From Keywords to Meaning

Before I show you how to embed text, I want to show you why the obvious approach fails.

Imagine we have our 7 product chunks above and a customer asks:

> "Does Barclays charge me when I spend money abroad?"

The word **"abroad"** does not appear in any of our chunks. The phrase **"foreign transaction fee"** appears only in the Rewards card chunk. A naive keyword search would miss this match unless the customer used the exact phrase from the doc.

Let me show you that problem first.

In [ ]:
# NAIVE DEMO - keyword search.
# Watch what happens when we search for customer intent using simple string matching.

def keyword_search(query: str, docs: list, top_k: int = 3) -> list:
    """Naive keyword search: count how many query words appear in each doc."""
    query_words = set(query.lower().split())
    scores = []
    for i, doc in enumerate(docs):
        doc_words = set(doc.lower().split())
        # Count overlap between query words and document words.
        overlap = len(query_words & doc_words)
        scores.append((overlap, i, doc))
    # Sort by overlap score, descending.
    scores.sort(reverse=True)
    return scores[:top_k]


# Test with a question that uses different vocabulary than our docs.
test_query = "Does Barclays charge me when I spend money abroad?"

print(f"Query: {test_query}")
print()
print("Keyword search results (score = word overlap):")
results = keyword_search(test_query, chunks)
for score, idx, doc in results:
    print(f"  score={score}  [{idx}] {doc[:80]}...")

print()
print("Problem: 'abroad' and 'spend money' are not in the chunks.")
print("The Rewards card chunk (which mentions 'foreign transaction fee') may not rank first.")
print("Semantic meaning is lost - keyword search cannot bridge synonyms.")

### What is an Embedding?

An embedding is a dense numeric vector that captures the *meaning* of a piece of text. Similar meanings land close together in vector space, even if the surface words are different.

"Spend money abroad" and "foreign transaction fee" end up near each other in the embedding space because the embedding model was trained on billions of documents where these phrases appear in similar contexts.

```mermaid
graph TB
    subgraph Input["Input texts (raw strings)"]
        T1["Text A:<br/>'spend money abroad'"]
        T2["Text B:<br/>'foreign transaction fee'"]
        T3["Text C:<br/>'mortgage rate 4.2 percent for 60 percent LTV'"]
    end

    Model["text-embedding-3-small<br/>(OpenAI Embeddings API)"]

    T1 --> Model
    T2 --> Model
    T3 --> Model

    Model --> V1["Vector A<br/>1536 floats<br/>[0.12, -0.04, 0.31, ...]"]
    Model --> V2["Vector B<br/>1536 floats<br/>[0.15, -0.06, 0.29, ...]"]
    Model --> V3["Vector C<br/>1536 floats<br/>[-0.41, 0.22, -0.18, ...]"]

    subgraph Space["Semantic vector space (1536 dimensions)"]
        direction LR
        V1 -.->|"cosine sim approx 0.85<br/><b>HIGH similarity</b><br/>same meaning"| V2
        V2 -.->|"cosine sim approx 0.10<br/><b>LOW similarity</b><br/>unrelated topic"| V3
    end

    style T1 fill:#cce5ff
    style T2 fill:#cce5ff
    style T3 fill:#fff5cc
    style V1 fill:#ccffcc
    style V2 fill:#ccffcc
    style V3 fill:#ffcccc
    style Model fill:#e0e0ff,stroke:#3333aa,stroke-width:2px
```

Key facts about `text-embedding-3-small`:

- Output: a Python list of 1536 floats (each value roughly between -1 and 1)
- Distance metric we use: cosine similarity (angle between vectors, not Euclidean distance)
- Cosine similarity of 1.0 means identical meaning, 0.0 means orthogonal, -1.0 means opposite
- This model is fast, cheap, and accurate for retrieval tasks
- Do NOT confuse it with `gpt-4o`: the embeddings model produces a vector, NOT a text answer

In [ ]:
# FULL WORKING DEMO - calling the OpenAI Embeddings API.
# I'm going to show you the exact function we will use throughout the topic.

def embed_texts(texts: list) -> list:
    """
    Embed a list of strings using text-embedding-3-small.
    Returns a list of lists (one float list per input text).

    The API accepts a list of strings in a single call (batch embedding).
    That is much more efficient than calling once per text.
    """
    # client.embeddings.create is the correct call for the Embeddings API.
    # Do NOT use client.chat.completions.create here - that one is for text generation.
    response = client.embeddings.create(
        model=EMBED_MODEL,   # text-embedding-3-small (1536 dimensions)
        input=texts          # list of strings - batched in one API call
    )

    # response.data is a list of Embedding objects, one per input string.
    # Each Embedding object has a .embedding attribute that is already a Python list of floats.
    # We do NOT need np.array() here - ChromaDB accepts plain Python lists.
    embeddings = [e.embedding for e in response.data]

    return embeddings


# Test with two semantically similar phrases and one unrelated phrase.
test_texts = [
    "Does Barclays charge a fee when I spend abroad?",         # customer query
    "Foreign transaction fee: 0 percent.",                      # doc-style chunk - should be close
    "The mortgage rate is 4.2 percent for 60 percent LTV.",    # unrelated - should be far
]

test_embeddings = embed_texts(test_texts)

print(f"Number of embeddings returned: {len(test_embeddings)}")
print(f"Dimensions per embedding:      {len(test_embeddings[0])}")
print(f"First 5 values of embedding 0: {[round(v, 4) for v in test_embeddings[0][:5]]}")
print()

# Quick cosine similarity check using numpy.
import numpy as np

def cosine_similarity(v1: list, v2: list) -> float:
    """Compute cosine similarity between two vectors."""
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


sim_query_card     = cosine_similarity(test_embeddings[0], test_embeddings[1])
sim_query_mortgage = cosine_similarity(test_embeddings[0], test_embeddings[2])

print(f"Similarity: [query] vs [foreign fee chunk]:  {sim_query_card:.4f}  (expected: high)")
print(f"Similarity: [query] vs [mortgage chunk]:     {sim_query_mortgage:.4f}  (expected: lower)")
print()
print("The embedding model correctly places 'abroad spending' near 'foreign transaction fee'.")

## Lab 1: Implement embed_texts

Now it's your turn. I want you to implement the `embed_texts` function from scratch.

**Your task**: Fill in the `None` placeholders in the starter cell below. The function signature and docstring are already written for you.

**Stretch** (if you finish early): Add a check that raises a `ValueError` if any string in the input list is empty (empty strings cause API errors).

**Time**: 15-20 minutes.

In [ ]:
# Lab 1 SOLUTION: Implement embed_texts
# Goal: convert a list of strings into a list of dense embedding vectors via the OpenAI API.

def embed_texts(texts: list) -> list:
    """
    Embed a list of strings using text-embedding-3-small.
    Returns a list of float lists (one per input text).
    """
    # SOLUTION step 1: call the embeddings API with our chosen model and the list of strings.
    # Sending all texts in a single call (batching) is much faster than one call per text
    # because it avoids the per-request overhead and amortises HTTP latency.
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )

    # SOLUTION step 2: response.data is a list of Embedding objects, one per input text.
    # Each object has .embedding which is already a Python list of floats - no numpy
    # conversion needed because ChromaDB accepts plain Python lists.
    embeddings = [e.embedding for e in response.data]

    return embeddings


# Verification - this should return one 1536-dim vector per chunk.
lab1_embeddings = embed_texts(chunks)

if lab1_embeddings is not None and len(lab1_embeddings) == len(chunks):
    print(f"[OK] embed_texts returned {len(lab1_embeddings)} embeddings.")
    print(f"[OK] Each embedding has {len(lab1_embeddings[0])} dimensions.")
    print("Lab 1 complete!")
else:
    print("[  ] embed_texts returned None or wrong count - check your code above.")

In [ ]:
# Safety-net: if you did not finish Lab 1, run this cell so the rest of
# the notebook still works. If you DID finish Lab 1, skip this cell.
try:
    if lab1_embeddings is None or len(lab1_embeddings) != len(chunks):
        raise ValueError("lab1_embeddings not set correctly")
    print("Lab 1 output looks correct - safety-net not needed. Skipping.")
except (NameError, ValueError):
    print("Using safety-net: calling the embeddings API with the instructor implementation.")
    _sn_response = client.embeddings.create(
        model=EMBED_MODEL,
        input=chunks
    )
    lab1_embeddings = [e.embedding for e in _sn_response.data]
    print(f"Safety-net produced {len(lab1_embeddings)} embeddings of dim {len(lab1_embeddings[0])}.")

## Concept 2: ChromaDB - Building a Vector Store

We now have 7 embedding vectors. But storing them in a Python list and doing cosine similarity manually is slow at scale and throws away all the index structure that a real vector store gives us.

Let me show you what that brute-force approach looks like before we move to a real vector store.

In [ ]:
# NAIVE APPROACH - brute-force cosine search over a plain Python list.
# This works for 7 docs but falls apart at 7,000 or 700,000 docs.

def brute_force_retrieve(query: str, doc_embeddings: list, docs: list, top_k: int = 3) -> list:
    """Search by computing cosine similarity to every embedding. O(n) per query."""
    # Embed the query (one API call per query - already inefficient at scale).
    query_vec = embed_texts([query])[0]
    query_arr = np.array(query_vec)

    scores = []
    for i, doc_emb in enumerate(doc_embeddings):
        doc_arr = np.array(doc_emb)
        sim = float(np.dot(query_arr, doc_arr) / (np.linalg.norm(query_arr) * np.linalg.norm(doc_arr)))
        scores.append((sim, i, docs[i]))

    scores.sort(reverse=True)
    return scores[:top_k]


# This works fine for our 7 docs.
test_query = "Does Barclays charge me when I spend money abroad?"
naive_results = brute_force_retrieve(test_query, lab1_embeddings, chunks)

print(f"Query: {test_query}")
print()
print("Brute-force results:")
for sim, idx, doc in naive_results:
    print(f"  sim={sim:.4f}  [{idx}] {doc[:80]}...")

print()
print("This works - but we have to manage the embeddings list ourselves,")
print("re-embed on every restart, and scan every doc on every query.")
print("ChromaDB handles all of that for us, including disk persistence.")

### ChromaDB: a Persistent, Indexed Vector Store

ChromaDB stores your embeddings on disk and lets you query them with an HNSW index (Hierarchical Navigable Small World) - much faster than brute-force at scale.

```mermaid
graph TB
    subgraph AddCall["collection.add(ids=..., documents=..., embeddings=...)"]
        direction LR
        AID["ids list:<br/>['doc_0',<br/>'doc_1',<br/>'doc_2', ...]"]
        ADOC["documents list:<br/>['Personal Loan...',<br/>'Rewards Card...',<br/>'Savings Acct...', ...]"]
        AEMB["embeddings list:<br/>[[0.12, -0.04, ...],<br/>[0.15, -0.06, ...],<br/>[0.31, 0.22, ...], ...]"]
    end

    AddCall ==>|"populates 3 parallel arrays"| Coll

    subgraph Coll["ChromaDB collection 'barclays_products'<br/>(persisted on disk via PersistentClient)"]
        direction LR
        CID["<b>ids</b><br/>doc_0<br/>doc_1<br/>doc_2<br/>doc_3<br/>..."]
        CDOC["<b>documents</b><br/>(raw text)<br/>'Personal Loan...'<br/>'Rewards Card...'<br/>'Savings Acct...'<br/>'Mortgage...'<br/>..."]
        CEMB["<b>embeddings</b><br/>(1536-dim vectors)<br/>indexed via HNSW<br/>cosine distance<br/>configured at create"]
    end

    Q["collection.query(<br/>query_embeddings=[query_vec],<br/>n_results=3)"] ==>|"cosine search"| Coll

    Coll ==> R["result dict (nested by query):<br/>result['ids'][0]: top-3 ids<br/>result['documents'][0]: top-3 docs<br/>result['distances'][0]: top-3 cosine distances"]

    style AddCall fill:#cce5ff,stroke:#0044aa,stroke-width:2px
    style Q fill:#fff5cc,stroke:#aa7700,stroke-width:2px
    style Coll fill:#e5e5e5,stroke:#444444,stroke-width:2px
    style R fill:#ccffcc,stroke:#006600,stroke-width:2px
```

Key ChromaDB patterns for this course:

- `chromadb.PersistentClient(path="./chroma_db")` - saves the collection to disk so we don't lose it on kernel restart
- `client.get_or_create_collection(name, configuration={"hnsw": {"space": "cosine"}})` - idempotent creation
- `collection.add(ids=[...], documents=[...], embeddings=[...])` - add all three parallel lists at once
- `collection.query(query_embeddings=[vec], n_results=3)` - returns a nested dict
- `result["documents"][0]` - the matched text strings (note the outer list - it's a list of lists, one per query you sent)

The cosine configuration line is the one students most often get wrong. In ChromaDB 1.x the HNSW space config goes inside the `configuration` key, **not** a flat `metadata` dict like older docs may show.

In [ ]:
# FULL WORKING DEMO - building and querying a ChromaDB collection.

def add_to_store(collection, docs: list, embeddings: list) -> None:
    """
    Add documents and their embeddings to a ChromaDB collection.
    Uses sequential IDs: doc_0, doc_1, ...
    """
    ids = [f"doc_{i}" for i in range(len(docs))]

    # collection.add takes parallel lists: ids, documents (raw text), embeddings (vectors).
    # The documents list is stored alongside the vectors so we can return raw text at query time.
    collection.add(
        ids=ids,
        documents=docs,
        embeddings=embeddings
    )
    print(f"Added {len(docs)} documents to the collection.")


def retrieve(query: str, collection, n_results: int = 3) -> list:
    """
    Embed the query and return the top-n matching document strings.
    """
    # Embed the query text into a vector. embed_texts returns a list - take [0].
    query_vec = embed_texts([query])[0]

    # Query the collection. Note: query_embeddings takes a LIST of vectors (batch queries),
    # so we wrap our single query vector in a list.
    result = collection.query(
        query_embeddings=[query_vec],
        n_results=n_results
    )

    # result["documents"] is a list of lists: one inner list per query we sent.
    # We sent one query, so we take result["documents"][0] to get our list of matched strings.
    matched_docs = result["documents"][0]

    return matched_docs


# Build the collection.
# PersistentClient saves to ./chroma_db so we do not lose data if the kernel restarts.
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# get_or_create_collection is idempotent: safe to re-run the cell.
# The cosine distance config goes via "configuration" -> "hnsw" -> "space" in ChromaDB 1.x.
collection = chroma_client.get_or_create_collection(
    name="barclays_products",
    configuration={"hnsw": {"space": "cosine"}}
)

# Add our documents and the embeddings produced in Lab 1.
add_to_store(collection, chunks, lab1_embeddings)

# Query the collection.
test_query = "Does Barclays charge me when I spend money abroad?"
top_chunks = retrieve(test_query, collection, n_results=3)

print()
print(f"Query: {test_query}")
print()
print(f"Top {len(top_chunks)} retrieved chunks:")
for i, chunk in enumerate(top_chunks):
    print(f"  [{i+1}] {chunk[:100]}...")

## Lab 2: Implement retrieve

Now it's your turn. Implement the `retrieve` function that embeds a query and returns the top matching documents from the collection.

**Your task**: Fill in the `None` placeholders in the starter cell below.

**Stretch** (if you finish early): Modify your `retrieve` function to also return the cosine distance scores alongside each document. Hint: add `include=["distances"]` to the query call and return a list of `(doc, score)` tuples.

**Time**: 15-20 minutes.

In [ ]:
# Lab 2 SOLUTION: Implement retrieve
# Goal: turn a natural-language query into the top-n matching document strings.

def retrieve(query: str, collection, n_results: int = 3) -> list:
    """
    Embed the query and return the top-n matching document strings from the collection.
    Returns a plain list of strings.
    """
    # SOLUTION step 1: embed_texts always takes a list and returns a list of vectors.
    # Even for a single query we wrap it in [query] and then take index [0] to unwrap.
    query_vec = embed_texts([query])[0]

    # SOLUTION step 2: query_embeddings expects a LIST of vectors because the API supports
    # batch queries. We have one query so we pass [query_vec]. ChromaDB returns a dict.
    result = collection.query(
        query_embeddings=[query_vec],
        n_results=n_results
    )

    # SOLUTION step 3: result["documents"] is a list of lists - one inner list per query
    # vector we sent. We sent one, so [0] gives us the matched strings for our query.
    # Common bug: forgetting [0] returns a list of lists instead of a flat list.
    matched_docs = result["documents"][0]

    return matched_docs


# Verification.
lab2_query = "What is the interest rate on a Barclays savings account?"
lab2_results = retrieve(lab2_query, collection, n_results=2)

if lab2_results is not None and len(lab2_results) > 0:
    print(f"[OK] retrieve returned {len(lab2_results)} results.")
    for i, doc in enumerate(lab2_results):
        print(f"  [{i+1}] {doc[:90]}...")
    print("Lab 2 complete!")
else:
    print("[  ] retrieve returned None or empty - check your steps above.")

In [ ]:
# Safety-net: if you did not finish Lab 2, run this cell so the rest of
# the notebook still works. If you DID finish Lab 2, skip this cell.
try:
    _test = retrieve("test", collection, n_results=1)
    if _test is None or len(_test) == 0:
        raise ValueError("retrieve not working")
    print("Lab 2 retrieve works - safety-net not needed. Skipping.")
except Exception:
    print("Using safety-net: restoring the retrieve function from the demo.")

    def retrieve(query: str, collection, n_results: int = 3) -> list:
        """Safety-net implementation of retrieve - identical to the demo above."""
        qv = embed_texts([query])[0]
        result = collection.query(query_embeddings=[qv], n_results=n_results)
        return result["documents"][0]

    print("Safety-net retrieve function restored.")

## Concept 3: The End-to-End RAG Pipeline

We have embeddings. We have a vector store. We have retrieval. Now I want to show you what happens when you skip all of that and just ask the model a product question cold.

In [ ]:
# NAIVE DEMO - answering without retrieval.
# This is what the chatbot from Topics 3-5 does: no product context, just the model's training memory.

no_rag_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a helpful Barclays customer service assistant."
        },
        {
            "role": "user",
            "content": "What is the overdraft limit on the Barclays Student Account in year 2?"
        }
    ],
    temperature=0.2,
    max_tokens=200
)

print("Answer WITHOUT retrieval (model guessing from training data):")
print(no_rag_response.choices[0].message.content)
print()
print("The model may give a number, but it could be outdated or wrong - it has no source.")
print("With RAG, we inject the actual chunk that mentions 'up to 1,000 GBP from year 2 onwards'.")

### The Full RAG Pipeline

```mermaid
graph TD
    Q["<b>Customer query</b><br/>'Does Barclays charge me<br/>for spending abroad?'"]

    Q --> E1["Step 1: embed query<br/>embed_texts([query])<br/>via text-embedding-3-small"]
    E1 --> QV["Query vector<br/>(1536 floats)"]

    QV --> S1["Step 2: cosine search<br/>collection.query(<br/>query_embeddings=[query_vec],<br/>n_results=3)"]

    DB[("ChromaDB collection<br/>'barclays_products'<br/>7 doc chunks indexed<br/>cosine distance, HNSW")]
    DB -.->|"semantic match<br/>via HNSW index"| S1

    S1 --> Chunks["Top-3 retrieved chunks:<br/>1. 'Rewards Card: foreign txn fee 0 percent'<br/>2. 'Travel Pack: zero foreign fees'<br/>3. 'Student Account: ...'"]

    Chunks --> P["Step 3: build augmented prompt"]

    subgraph Prompt["Prompt structure (the 'A' in RAG)"]
        direction TB
        SYS["<b>system</b>:<br/>'You are a Barclays product expert.<br/>Answer ONLY from the context below.<br/>If not in context, say so.'"]
        USR["<b>user</b>:<br/>'Context:<br/>{chunks joined by blank lines}<br/><br/>Question: {customer query}'"]
    end

    P --> SYS
    P --> USR

    SYS --> GEN["Step 4: generate answer<br/>client.chat.completions.create(<br/>model='gpt-4o',<br/>messages=[system, user],<br/>temperature=0.2)"]
    USR --> GEN

    GEN --> A["<b>Grounded answer</b><br/>'No, the Barclays Rewards card has<br/>no foreign transaction fee (0 percent).<br/>The Travel Pack also includes<br/>zero foreign transaction fees.'"]

    style Q fill:#fff5cc,stroke:#aa7700,stroke-width:2px
    style QV fill:#cce5ff
    style DB fill:#e0e0ff,stroke:#3333aa,stroke-width:2px
    style Chunks fill:#cce5ff
    style Prompt fill:#ffe5cc,stroke:#aa5500,stroke-width:2px
    style GEN fill:#e0e0ff,stroke:#3333aa,stroke-width:2px
    style A fill:#ccffcc,stroke:#006600,stroke-width:3px
```

The two-sentence rule for the RAG prompt:

- **System message**: assign the role AND say "answer only from the provided context"
- **User message**: `"Context:\n{chunks joined}\n\nQuestion: {query}"`

This structure keeps the model anchored to what we retrieved and prevents it from supplementing answers with training-data guesses.

In [ ]:
# FULL WORKING DEMO - the complete RAG pipeline.

def rag_answer(query: str, collection, n_results: int = 3) -> str:
    """
    Full RAG pipeline: embed query, retrieve top chunks, augment prompt, call gpt-4o.
    Returns the generated answer string.
    """
    # Step 1: retrieve the most relevant document chunks.
    relevant_chunks = retrieve(query, collection, n_results=n_results)

    # Step 2: build the augmented prompt.
    # Join chunks with a blank line separator so the model sees them as distinct passages.
    context_block = "\n\n".join(relevant_chunks)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful Barclays product knowledge assistant. "
                "Answer the customer's question using ONLY the context provided below. "
                "If the context does not contain the answer, say: "
                "'I don't have that information in the product documentation.'"
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context_block}\n\nQuestion: {query}"
        }
    ]

    # Step 3: call gpt-4o with the augmented prompt.
    # Low temperature for factual, reproducible answers grounded in the context.
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
        max_tokens=300
    )

    return response.choices[0].message.content


# Test the full pipeline on three different product questions.
test_questions = [
    "What is the overdraft limit on the Barclays Student Account in year 2?",
    "Does Barclays charge a fee when I spend money abroad on the Rewards card?",
    "What is the AER on the Barclays savings account?",
]

for q in test_questions:
    answer = rag_answer(q, collection)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print()

## Lab 3: Build Your Own RAG Pipeline

This is the Day 2 **open-ended** lab. There are no step-by-step hints and no `None` placeholders. Your task is to build a complete RAG pipeline from scratch, using what you have learned today.

**Your task**: Implement the function `build_rag_assistant` below. The docstring describes exactly what it should do. The function signature is given to you. *How* you implement it is up to you.

You may use `embed_texts`, `retrieve`, `add_to_store`, or any helpers you have built today. You can also swap in your Topic 2 `chunks` list for a richer document set.

**Time**: 25-30 minutes (no time limit - take it home if you need to).

In [ ]:
# Lab 3 SOLUTION (Tier 3 - open ended): Build a complete RAG assistant from scratch.
# This is one valid reference implementation - many other shapes are equally correct.

def build_rag_assistant(docs: list, query: str, collection_name: str = "lab3_collection") -> dict:
    """
    Build a complete single-shot RAG assistant from scratch.
    See the exercise cell above for the full requirements docstring.
    """
    # SOLUTION step 1: build a fresh ChromaDB collection with cosine distance.
    # PersistentClient writes to ./chroma_db so the collection survives kernel restarts.
    # get_or_create_collection is idempotent - safe to call repeatedly with the same name.
    lab3_chroma = chromadb.PersistentClient(path="./chroma_db")
    lab3_collection = lab3_chroma.get_or_create_collection(
        name=collection_name,
        configuration={"hnsw": {"space": "cosine"}}
    )

    # SOLUTION step 2: embed every document in one batched API call.
    # Using the embed_texts function from Lab 1 keeps this concise and consistent.
    doc_embeddings = embed_texts(docs)

    # SOLUTION step 3: add docs and embeddings together with sequential ids.
    # If the collection already has docs from a previous run, .add will raise on duplicate ids.
    # We guard with try/except so the function is re-runnable in a notebook.
    ids = [f"lab3_doc_{i}" for i in range(len(docs))]
    try:
        lab3_collection.add(ids=ids, documents=docs, embeddings=doc_embeddings)
    except Exception:
        # Already populated from a prior run - safe to ignore for this lab.
        pass

    # SOLUTION step 4: retrieve top-3 chunks for the customer query.
    # Reuse the retrieve helper from Lab 2 - it embeds the query and returns matched strings.
    retrieved_chunks = retrieve(query, lab3_collection, n_results=3)

    # SOLUTION step 5: build the augmented prompt and call gpt-4o.
    # The system message hard-grounds the assistant: "answer only from the provided context".
    # This is the line that prevents the model from supplementing with training-data guesses.
    context_block = "\n\n".join(retrieved_chunks)
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful Barclays product knowledge assistant. "
                "Answer the customer's question using ONLY the context provided below. "
                "If the context does not contain the answer, say: "
                "'I don't have that information in the product documentation.'"
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context_block}\n\nQuestion: {query}"
        }
    ]
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
        max_tokens=300
    )
    answer = response.choices[0].message.content

    # SOLUTION step 6: return all three required keys.
    # collection_count uses .count() so the caller can verify the collection was populated.
    return {
        "answer": answer,
        "retrieved_chunks": retrieved_chunks,
        "collection_count": lab3_collection.count(),
    }


# Quick smoke test of the reference solution.
result = build_rag_assistant(
    docs=chunks,
    query="Does the Barclays Travel Pack include airport lounge access?",
)
print(f"Answer: {result['answer']}")
print(f"Collection count: {result['collection_count']}")
print(f"Retrieved {len(result['retrieved_chunks'])} chunks for the query.")

## Topic 6 Wrap-Up: What We Built Today

Let me recap what we built and where it fits.

**What you built today**:

- `embed_texts(texts)` - converts any list of strings to dense vectors via `text-embedding-3-small`
- A ChromaDB `PersistentClient` collection with cosine distance configured
- `retrieve(query, collection, n_results)` - semantic search that returns the top-k chunks
- `rag_answer(query, collection)` - the full pipeline: embed, retrieve, augment, generate
- `build_rag_assistant(docs, query)` (Lab 3) - your own end-to-end implementation

**The Customer Service Assistant flow now looks like this**:

```text
customer_question
      |
      v
[embed_texts([question])]   -> query vector
      |
      v
[collection.query(...)]     -> top-3 relevant chunks
      |
      v
[augmented prompt: context + question]
      |
      v
[gpt-4o via chat.completions.create]
      |
      v
grounded answer (only from the docs we gave it)
```

**Key takeaways**:

- Embeddings capture *meaning*, not just words - semantic search bridges synonyms
- ChromaDB stores vectors on disk with HNSW indexing - fast and persistent
- The cosine config line is the most common student mistake - always check it
- Prompt structure matters: "answer ONLY from context" keeps the model grounded
- `result["documents"][0]` is a nested list - always take `[0]` after a single-query `query()` call

**Connection to Topic 7 (Advanced RAG and Web Search)**:

Our vector store holds static product docs. But Barclays rates change - the savings AER we embedded today may be stale next month. In Topic 7 we add a **web search fallback** so the assistant can fetch live information when the vector store does not have a confident match.

**Homework extensions** (optional):

1. Swap in your Topic 2 `chunks` list (from the real Barclays PDF extraction) and rebuild the collection. Does retrieval quality improve with more granular chunks?
2. Add a confidence threshold to `retrieve`: if the best cosine similarity score is below 0.5, return an empty list and have `rag_answer` say "I cannot find relevant information."
3. Implement multi-query retrieval: generate 3 rephrased versions of the customer question using `gpt-4o`, embed all 3, retrieve top-2 for each, deduplicate, then generate the answer from the combined unique chunks. Does this improve recall?